## Title
Estimating the Causal Effect of Discounts on Product Sales in E-Commerce Markets

#### Aim

To determine whether offering price discounts causes a statistically significant increase in product sales, after controlling for factors such as product category, seasonality, rating, and baseline demand.
The project aims to separate true causal impact from mere correlation, helping businesses understand how much discounts actually drive demand

###### Importing libraries

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score
from scipy import stats

######  Loading dataset

In [9]:
data = pd.read_csv("Sample - Superstore.csv", encoding = "latin1")

In [10]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [11]:
#info
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [12]:
#making a copy
df = data.copy()

In [13]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

##### Data Preprocessing

In [14]:
# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

In [15]:
# Feature engineering
df['order_year'] = df['Order Date'].dt.year
df['order_month'] = df['Order Date'].dt.month
df['order_weekday'] = df['Order Date'].dt.weekday

In [16]:
# Treatment variable: binary treated = discount > 0
df['Treated'] = (df['Discount'] > 0).astype(int)

In [17]:
# capping extreme discounts (if any) - here just clip to [0,1]
df['Discount'] = df['Discount'].clip(lower=0, upper=1)

In [18]:
# Selecting covariates for propensity model
# using Quantity, Profit, Category, Sub-Category, Segment, Region, order_month, Ship Mode, order_year
covariates = ['Quantity', 'Profit', 'order_month', 'order_year']

In [19]:
# Creating dummies for categorical variables
cat_vars = ['Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode']
df = pd.get_dummies(df, columns=cat_vars, drop_first=True)


In [20]:
# Extending covariates list with created dummy columns
covariates += [c for c in df.columns if any(
    prefix in c for prefix in ['Category_', 'Sub-Category_', 'Segment_', 'Region_', 'Ship Mode_'])]


In [21]:
# Dropping rows with missing values in relevant columns
use_cols = ['Sales', 'Discount', 'Treated'] + covariates
df_model = df[use_cols].dropna().reset_index(drop=True)

In [22]:
print("Model sample size:", df_model.shape[0])


Model sample size: 9994


###### Propensity score estimation (logistic regression)

In [23]:

X = df_model[covariates].astype(float)
y = df_model['Treated'].astype(int)

In [24]:
# Scaling continuous covariates
scaler = StandardScaler()
X_scaled = X.copy()

# scaling only the numeric columns
X_scaled.iloc[:, :] = scaler.fit_transform(X_scaled)

logreg = LogisticRegression(max_iter=1000, solver='lbfgs')
logreg.fit(X_scaled, y)
pscore = logreg.predict_proba(X_scaled)[:, 1]
df_model['pscore'] = pscore

print("Propensity model AUC:", roc_auc_score(y, pscore).round(3))

Propensity model AUC: 0.804


This indicates the logistic regression predicting whether a sale received a discount (Treated) is pretty good. A value above 0.8 is considered strong

In [25]:
# Matching (1:1 nearest neighbor on propensity score)
# Separate treated and control indices
treated_idx = df_model[df_model['Treated'] == 1].index.to_numpy()
control_idx = df_model[df_model['Treated'] == 0].index.to_numpy()

# Build NN on control propensity scores
nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(df_model.loc[control_idx, ['pscore']])
distances, indices = nn.kneighbors(df_model.loc[treated_idx, ['pscore']])

# indices gives index positions within control_idx array
matched_control_idx = control_idx[indices.flatten()]

# Create matched DataFrame
matched_treated = df_model.loc[treated_idx].copy().reset_index(drop=True)
matched_control = df_model.loc[matched_control_idx].copy().reset_index(drop=True)

matched = pd.concat([matched_treated.assign(matched_group=range(len(matched_treated))),
                     matched_control.assign(matched_group=range(len(matched_control)))])

print("Matched sample size (treated + control):", matched.shape[0])


Matched sample size (treated + control): 10392


The above shows that there is a successfull matched between treated and control units 1:1 nearest neighbor, giving a balanced sample for causal estimation

In [26]:
# Balance diagnostics
def standardized_mean_diff(df_treat, df_control, var):
    mean_t = df_treat[var].mean()
    mean_c = df_control[var].mean()
    sd_pooled = np.sqrt((df_treat[var].var(ddof=1) + df_control[var].var(ddof=1)) / 2)
    if sd_pooled == 0:
        return 0.0
    return (mean_t - mean_c) / sd_pooled

# computing for a sample of covariates (numeric and some dummies)
cov_for_balance = covariates 
smds_before = {}
smds_after = {}

treated_full = df_model[df_model['Treated'] == 1]
control_full = df_model[df_model['Treated'] == 0]


In [27]:
for v in cov_for_balance:
    smd_before = standardized_mean_diff(treated_full, control_full, v)
    smd_after = standardized_mean_diff(matched_treated, matched_control, v)
    smds_before[v] = smd_before
    smds_after[v] = smd_after

balance_df = pd.DataFrame({
    'covariate': list(smds_before.keys()),
    'smd_before': list(smds_before.values()),
    'smd_after': list(smds_after.values())
}).sort_values('smd_before', key=lambda s: s.abs(), ascending=False)

print(balance_df.head(15).round(3))

                   covariate  smd_before  smd_after
8       Sub-Category_Binders       0.455      0.009
1                     Profit      -0.317     -0.313
10       Sub-Category_Chairs       0.277     -0.025
17        Sub-Category_Paper      -0.233      0.035
4   Category_Office Supplies      -0.194      0.096
21       Sub-Category_Tables       0.188      0.135
19      Sub-Category_Storage      -0.178      0.064
7           Sub-Category_Art      -0.171      0.019
18       Sub-Category_Phones       0.164     -0.031
26               Region_West      -0.153      0.380
14  Sub-Category_Furnishings      -0.152      0.040
15       Sub-Category_Labels      -0.137      0.043
9     Sub-Category_Bookcases       0.134      0.033
16     Sub-Category_Machines       0.100     -0.171
6    Sub-Category_Appliances      -0.090      0.035


###### Estimate ATT using matched pairs

In [28]:
matched_sorted = matched.sort_values('matched_group')

# for each group, compute difference in Sales (treated - control)
grouped = matched_sorted.groupby('matched_group')
diffs = grouped.apply(lambda g: g.loc[g['Treated'] == 1, 'Sales'].values[0] - g.loc[g['Treated'] == 0, 'Sales'].values[0])
ATT = diffs.mean()

# computing standard error (bootstrap or sample std / sqrt(n))
se_ATT = diffs.std(ddof=1) / np.sqrt(len(diffs))
tstat = ATT / se_ATT

print(f"ATT (matched, treated>0 vs control): {ATT:.3f}")
print(f"SE: {se_ATT:.3f}, t-stat: {tstat:.3f}, N_pairs: {len(diffs)}")

ATT (matched, treated>0 vs control): 68.817
SE: 7.743, t-stat: 8.888, N_pairs: 5196


* Most covariates have SMD near 0 after matching, meaning the treated and control groups are well-balanced. Only a few, like Profit, remain slightly imbalanced, but this is minor.

* ATT (Average Treatment Effect on Treated): 68.817

* SE: 7.743

* t-stat: 8.888
* Shows that receiving a discount increases Sales by about 68.8 units on average, and the effect is statistically significant (t >> 2).

* Bootstrap 95% CI for ATT: [54.507, 84.702]
* We are 95% confident that the true effect of a discount on Sales lies between 54.5 and 84.7 units.

###### Interpretation:
Discounts have a positive and statistically significant causal effect on Sales. PSM suggests that, on average, a sale with a discount generates about $69 more in revenue compared to a similar sale without a discount.

###### Bootstrap for 95% CI

In [29]:
n_boot = 1000
boot_atts = []
rng = np.random.default_rng(42)
for _ in range(n_boot):
    sample_idx = rng.integers(0, len(diffs), len(diffs))
    boot_atts.append(diffs.values[sample_idx].mean())
ci_lower = np.percentile(boot_atts, 2.5)
ci_upper = np.percentile(boot_atts, 97.5)
print(f"Bootstrap 95% CI for ATT: [{ci_lower:.3f}, {ci_upper:.3f}]")

Bootstrap 95% CI for ATT: [54.507, 84.702]


We are 95% confident that the true effect of a discount on Sales lies between 54.5 and 84.7 units

###### Causal Regression

In [34]:
# Clean df and df_model column names
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.replace("-", "_")

df_model.columns = df_model.columns.str.replace(" ", "_")
df_model.columns = df_model.columns.str.replace("-", "_")

# Ensure reg_df inherits cleaned names
reg_df = df_model.copy()

# Build formula dynamically
formula = (
    'Sales ~ Discount + Quantity + Profit + order_month + order_year + '
    + ' + '.join([
        c for c in reg_df.columns
        if c.startswith((
            'Category_', 
            'Sub_Category_', 
            'Segment_', 
            'Region_', 
            'Ship_Mode_'
        ))
    ])
)

print("Using formula:\n", formula)


Using formula:
 Sales ~ Discount + Quantity + Profit + order_month + order_year + Category_Office_Supplies + Category_Technology + Sub_Category_Appliances + Sub_Category_Art + Sub_Category_Binders + Sub_Category_Bookcases + Sub_Category_Chairs + Sub_Category_Copiers + Sub_Category_Envelopes + Sub_Category_Fasteners + Sub_Category_Furnishings + Sub_Category_Labels + Sub_Category_Machines + Sub_Category_Paper + Sub_Category_Phones + Sub_Category_Storage + Sub_Category_Supplies + Sub_Category_Tables + Segment_Corporate + Segment_Home_Office + Region_East + Region_South + Region_West + Ship_Mode_Same_Day + Ship_Mode_Second_Class + Ship_Mode_Standard_Class


In [35]:
model = smf.ols(formula=formula, data=reg_df).fit(cov_type='HC1')
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.412
Model:                            OLS   Adj. R-squared:                  0.410
Method:                 Least Squares   F-statistic:                     225.5
Date:                Tue, 09 Dec 2025   Prob (F-statistic):               0.00
Time:                        13:02:20   Log-Likelihood:                -75837.
No. Observations:                9994   AIC:                         1.517e+05
Df Residuals:                    9964   BIC:                         1.520e+05
Df Model:                          29                                         
Covariance Type:                  HC1                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

C:\Users\JAMES TECH\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 31, but rank is 30
  warnings.warn('covariance of constraints does not have full '


* R-squared: 0.412
* About 41% of the variation in Sales is explained by the model, which is decent for sales data.

* Discount coefficient: 77.57 (std err 64.16, p = 0.227)
Not statistically significant at 5% level (p > 0.05).

###### Other significant variables:

* Quantity: 48.92 (p < 0.001)

* Profit: 1.16 (p < 0.001)

* Sub_Category_Copiers, Sub_Category_Machines, Sub_Category_Phones are also significant.

###### Notes on multicollinearity:

Warning about smallest eigenvalue and covariance of constraints not full rank.

This suggests some dummies are highly correlated, causing instability in coefficient estimates, especially Discount.

###### Interpretation:

When controlling for all covariates and dummies, the Discount effect becomes statistically insignificant.

The magnitude (77.57) is similar to the PSM ATT (68.8), but OLS suffers from multicollinearity, reducing confidence in p-values.

This indicates that the true effect is likely positive, but the regression struggles to isolate it from correlated factors like Category, Segment, and Profit.